## Question 1

In [ ]:
%pip -q install torch torchvision pillow matplotlib tqdm

In [ ]:
import random
import time
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.models.detection import (
    FasterRCNN_MobileNet_V3_Large_320_FPN_Weights,
    fasterrcnn_mobilenet_v3_large_320_fpn,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_iou
from torchvision.transforms import functional as F
from tqdm.auto import tqdm

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_TRAIN_IMAGES = 120 if device.type == "cuda" else 32
EPOCHS = 3 if device.type == "cuda" else 1
BATCH_SIZE = 2 if device.type == "cuda" else 1
print("Device:", device)

In [ ]:
DATA_DIR = Path("data")
DATASET_DIR = DATA_DIR / "PennFudanPed"
ARCHIVE_PATH = DATA_DIR / "PennFudanPed.zip"
DATA_URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"

DATA_DIR.mkdir(parents=True, exist_ok=True)
if not (DATASET_DIR / "PNGImages").exists():
    if not ARCHIVE_PATH.exists():
        urlretrieve(DATA_URL, ARCHIVE_PATH)
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(DATA_DIR)
print("Dataset:", DATASET_DIR)

In [ ]:
class PennFudanDataset(Dataset):
    def __init__(self, root):
        self.root = Path(root)
        self.images = sorted((self.root / "PNGImages").glob("*.png"))
        self.masks = sorted((self.root / "PedMasks").glob("*.png"))
        assert len(self.images) == len(self.masks) > 0

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = Image.open(self.images[index]).convert("RGB")
        mask = np.array(Image.open(self.masks[index]))
        ids = np.unique(mask)
        ids = ids[ids != 0]

        boxes = []
        for object_id in ids:
            yy, xx = np.where(mask == object_id)
            if len(xx):
                boxes.append([xx.min(), yy.min(), xx.max() + 1, yy.max() + 1])

        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        target = {
            "boxes": boxes,
            "labels": torch.ones(len(boxes), dtype=torch.int64),
            "image_id": torch.tensor(index),
            "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),
            "iscrowd": torch.zeros(len(boxes), dtype=torch.int64),
        }
        return F.to_tensor(image), target

In [ ]:
dataset = PennFudanDataset(DATASET_DIR)
print("Images:", len(dataset))
print("Classes: background, pedestrian")

image, target = dataset[0]
print("Image shape:", tuple(image.shape))
print("Bounding boxes:", len(target["boxes"]))

In [ ]:
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(11, 6))
ax.imshow(image.permute(1, 2, 0))
for box in target["boxes"]:
    x1, y1, x2, y2 = box.tolist()
    ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                           fill=False, edgecolor="lime", linewidth=2))
    ax.text(x1, y1, "Person", color="white", bbox={"facecolor": "green", "alpha": 0.7})
ax.axis("off")
plt.show()

In [ ]:
indices = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(SEED)).tolist()
train_count = min(MAX_TRAIN_IMAGES, int(0.8 * len(indices)))
test_count = min(30, len(indices) - train_count)
train_dataset = Subset(dataset, indices[:train_count])
test_dataset = Subset(dataset, indices[-test_count:])
print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False,
                         collate_fn=collate_fn, num_workers=0)

In [ ]:
weights = FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT
model = fasterrcnn_mobilenet_v3_large_320_fpn(
    weights=weights, trainable_backbone_layers=0
)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)
model = model.to(device)
print(type(model).__name__)

In [ ]:
optimizer = torch.optim.SGD(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=0.002, momentum=0.9, weight_decay=0.0005
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    epoch_loss = 0.0
    for images, targets in tqdm(loader, desc="Training", leave=False):
        images = [im.to(device) for im in images]
        targets = [{key: value.to(device) for key, value in t.items()} for t in targets]
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

In [ ]:
loss_history = []
for epoch in range(EPOCHS):
    avg_loss = train_one_epoch(model, train_loader, optimizer, device)
    scheduler.step()
    loss_history.append(avg_loss)
    print(f"Epoch {epoch + 1}/{EPOCHS}: loss={avg_loss:.4f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.xticks(range(1, EPOCHS + 1))
plt.grid(alpha=0.3)
plt.show()

In [ ]:
model.eval()
sample_image, sample_target = test_dataset[0]
with torch.inference_mode():
    sample_prediction = model([sample_image.to(device)])[0]
sample_prediction = {key: value.cpu() for key, value in sample_prediction.items()}
print("Detections:", int((sample_prediction["scores"] >= 0.5).sum()))

In [ ]:
def draw_detections(image, boxes, scores=None, threshold=0.5, title=None):
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.imshow(image.permute(1, 2, 0))
    for i, box in enumerate(boxes):
        confidence = 1.0 if scores is None else float(scores[i])
        if confidence < threshold:
            continue
        x1, y1, x2, y2 = [float(value) for value in box]
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                               fill=False, edgecolor="lime", linewidth=2))
        ax.text(x1, y1, f"Person {confidence:.2f}", color="white",
                bbox={"facecolor": "green", "alpha": 0.7})
    ax.set_title(title or "Detections")
    ax.axis("off")
    plt.show()

In [ ]:
draw_detections(
    sample_image, sample_prediction["boxes"], sample_prediction["scores"],
    threshold=0.5, title="Faster R-CNN: pedestrian detection"
)

In [ ]:
model.eval()
predictions = []
ground_truths = []
with torch.inference_mode():
    for images, targets in tqdm(test_loader, desc="Evaluating"):
        output = model([images[0].to(device)])[0]
        predictions.append({key: value.cpu() for key, value in output.items()})
        ground_truths.append(targets[0])

In [ ]:
def average_precision_at_50(predictions, ground_truths):
    detections = []
    n_gt = sum(len(target["boxes"]) for target in ground_truths)
    matched = [set() for _ in ground_truths]

    for image_id, prediction in enumerate(predictions):
        for box, score in zip(prediction["boxes"], prediction["scores"]):
            detections.append((float(score), image_id, box.unsqueeze(0)))
    detections.sort(reverse=True, key=lambda item: item[0])

    tp = np.zeros(len(detections))
    fp = np.zeros(len(detections))
    for i, (_, image_id, box) in enumerate(detections):
        gt_boxes = ground_truths[image_id]["boxes"]
        if len(gt_boxes) == 0:
            fp[i] = 1
            continue
        overlaps = box_iou(box, gt_boxes).squeeze(0)
        best_match = int(overlaps.argmax())
        if float(overlaps[best_match]) >= 0.5 and best_match not in matched[image_id]:
            tp[i] = 1
            matched[image_id].add(best_match)
        else:
            fp[i] = 1

    precision = np.cumsum(tp) / np.maximum(np.cumsum(tp + fp), 1)
    recall = np.cumsum(tp) / max(n_gt, 1)
    ap = 0.0
    for r in np.linspace(0, 1, 101):
        ap += precision[recall >= r].max() if np.any(recall >= r) else 0
    return ap / 101, precision, recall

In [ ]:
ap50, precision_curve, recall_curve = average_precision_at_50(
    predictions, ground_truths
)
print(f"AP@0.50: {ap50:.4f}")

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(recall_curve, precision_curve)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-Recall Curve (AP@0.50={ap50:.3f})")
plt.xlim(0, 1)
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
n_timed = min(10, len(test_dataset))
with torch.inference_mode():
    model([test_dataset[0][0].to(device)])
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for i in range(n_timed):
        model([test_dataset[i][0].to(device)])
    if device.type == "cuda":
        torch.cuda.synchronize()

elapsed = time.perf_counter() - start
print(f"Average inference time: {elapsed / n_timed:.3f} seconds/image")
print(f"Inference FPS: {n_timed / elapsed:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6))
for i, ax in enumerate(axes):
    image, _ = test_dataset[i]
    ax.imshow(image.permute(1, 2, 0))
    prediction = predictions[i]
    for box, score in zip(prediction["boxes"], prediction["scores"]):
        if score < 0.5:
            continue
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                               fill=False, edgecolor="lime", linewidth=2))
        ax.text(x1, y1, f"Person {float(score):.2f}", fontsize=8,
                color="white", bbox={"facecolor": "green", "alpha": 0.7})
    ax.axis("off")
plt.tight_layout()
plt.savefig("lab08_detection_results.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "classes": ["background", "person"],
    "epochs": EPOCHS,
    "ap50": float(ap50),
}, "lab08_faster_rcnn_pedestrians.pth")
print("Model saved: lab08_faster_rcnn_pedestrians.pth")